# A Jansen leg: search widely, then polish, then check the Hessian
RMD project · the Jansen-linkage counterpart of `C13_leg_optimizer_v2.ipynb`

The code lives next to this notebook: `jansen_opt_torch.py` (synthesis), `verify_jansen.py` (independent NumPy check), `jansen_viz.py` (pictures). Run the cells in order **from this folder**.

The recipe is the four-bar one (SF §4.3–4.6) with two additions, because 27 numbers have many more valleys than 10:
a **global stage** (Latin hypercube sampling + differential evolution) picks the starts, and the **Hessian** classifies each minimum the polish finds.

In [ ]:
import os, numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import HTML
from torch.func import jacrev
import jansen_opt_torch as JO, jansen_viz as JV, verify_jansen as VJ
np.set_printoptions(precision=3, suppress=True)
print('torch', torch.__version__, '| device', JO.DEVICE)

## Step 0. The task in numbers
The group's 14 targets (`jansen_opt.py`), traced counter-clockwise. **Hybrid timing:** targets 0–3 lie on the floor and are **timed**: target $k$ is reached at crank travel $k\,\Delta\theta_s$ (constant ground speed). Targets 4–13 are **untimed**: the 11 gaps between them share the rest of the turn through a softmax, so they stay in order and the loop closes. Machine rule: the transmission angle at every dyad is at least 40° over the whole turn.

In [ ]:
JV.plot_task(); plt.show()

## Step 1. A candidate is a list of 27 numbers
Jansen's 13 letters (crank $m$, frame pivot $Z = (-a_x, -a_y)$, ten rods), the placement $(\theta_0, h_X, h_Y)$, and the timing $(\Delta\theta_s, q_1..q_{10})$.
There is no $\theta_1$: with $Z$ free in both directions, a body rotation is already $(a_x, a_y, \theta_0)$, so it would add an exact zero to the Hessian. For the same reason the first gap logit is pinned at 0.

In [ ]:
for n, (nm, lo, hi) in enumerate(zip(JO.NAMES, JO.DE_LO, JO.DE_HI)):
    s = 1 / JO.D2R if nm in ('theta0', 'dth_s') else 1
    print(f'{n:2d} {nm:7s} {lo * s:8.1f} .. {hi * s:7.1f}', '(deg)' if s != 1 else '')

## Step 2. Where is the foot?  Five circle intersections
$P_1$ (rods $j$, $b$), $P_2$ (triangle $b$-$d$-$e$), $P_3$ (rods $k$, $c$), $P_4$ (rods $f$, $g$), foot $F$ (triangle $g$-$h$-$i$). One branch sign per intersection; only $\sigma = (-,+,+,-,+)$ gives Jansen's leg.
Here are **Jansen's own numbers**, with only placement and timing fitted to $T$. `verify_jansen.report` re-solves the leg in NumPy at every 0.1°.

In [ ]:
JO.set_hand(+1)
v_holy = JO.holy_v().numpy()
VJ.report(v_holy, +1)
JV.plot_best(v_holy, +1, title="Jansen's holy numbers, placed on T"); plt.show()

In [ ]:
HTML(JV.animate_walk(v_holy, +1).to_jshtml())

Jansen's leg runs its lower dyad ($P_3$) to within about 1 mm of a toggle: $\mu_{min} \approx 9°$. Our 40° rule rejects it, so the optimizer has to find different proportions.

## Step 3. The misses, as one vector $\mathbf{r}$, and the cost $J = \mathbf{r}^T\mathbf{r}$
28 misses (X and Y of 14 targets, divided by $\sqrt{14}$, so $J$ = mean squared miss), 3 transmission-angle shortfalls (2 per degree), 3 assembly and 2 rigid-triangle penalties (sloped, 10 per mm, as in `leg_opt_torch.py`): **36 residuals**. The minimum over the turn is a soft minimum, so $J$ is smooth and its Hessian exists.

In [ ]:
r = JO.residuals_v(torch.as_tensor(v_holy))
print('J =', round(float(r @ r), 2), '| misses part', round(float(r[:28] @ r[:28]), 2))
print(dict(zip(JO.RES_NAMES[28:], np.round(r[28:].numpy(), 2))))

## Step 4. The Jacobian, exactly (autograd) and by nudging

In [ ]:
z = JO.to_z(torch.as_tensor(v_holy))
A = jacrev(JO.residuals)(z)
Afd = torch.stack([(JO.residuals(z + 1e-6 * e) - JO.residuals(z - 1e-6 * e)) / 2e-6 for e in torch.eye(27)], 1)
print('A is', tuple(A.shape), '| max |autograd - central difference| =', float((A - Afd).abs().max()))

## Step 5. Spread the starts evenly: Latin hypercube sampling
Each of the 27 ranges is cut into $n$ slices and every slice is used exactly once, so even 5000 samples cover every number's range evenly (`scipy.stats.qmc.LatinHypercube`). All 5000 are scored in one batched torch call.

In [ ]:
V = JO.lhs(5000, seed=0); Jl = JO.score(V)
print(f'best LHS sample J = {Jl.min():.1f} | median {np.median(Jl):.0f} | holy leg J = {float(r @ r):.1f}')
plt.figure(figsize=(7, 3)); plt.hist(np.log10(Jl), bins=60, color=JV.BLUE)
plt.axvline(np.log10(float(r @ r)), color=JV.PINK, label='holy leg'); plt.legend()
plt.xlabel('log10 J'); plt.ylabel('samples'); plt.title('5000 Latin hypercube designs'); plt.show()

## Step 6. Global search, then polish: the whole pipeline
For each hand (frame pivot left of the crank, as Jansen drew it, or mirrored):
1. **Differential evolution** (SciPy, a genetic-algorithm-family search) from the best LHS samples. The holy leg is injected into the first run only, so the other runs stay independent.
2. The final DE populations are pooled, and $K$ **distinct** starts (greedy, at least 3 % of the box apart) go to **batched Levenberg–Marquardt**.
3. The best are **finished with `least_squares`** in physical units. The sigmoid box of LM flattens next to a bound, and the best legs often sit on one.
4. Results are merged into basins, and the best are checked with the **Hessian** (Step 9).

This takes a few minutes on a CPU at the settings below. A full run, `python jansen_opt_torch.py`, saves `best_jansen_run.pt`; if that file is here, it is loaded instead.

In [ ]:
SETTINGS = dict(hands=(+1, -1), seeds=2, n_lhs=5000, popsize=15, maxiter=100, K=32, n_refine=8)
if os.path.exists('best_jansen_run.pt'):
    res = torch.load('best_jansen_run.pt', weights_only=False); print('loaded the saved full run')
else:
    res = JO.synthesize(**SETTINGS)
runs, basins, best = res['runs'], res['basins'], res['best']
JV.plot_de_convergence(runs); plt.show()

## Step 7. Many valleys: the basins found
Each row is a distinct minimum, with how many starts ended there and from which independent runs. As in SF §4.6, the best valley is not always the most common one.

In [ ]:
JO.basin_table(basins)
JV.plot_basins(basins); plt.show()

## Step 8. Verify the winner by analysis at every 0.1°, against every requirement

In [ ]:
VJ.report(best['v'], best['hand'])
JV.plot_best(best['v'], best['hand']); plt.show()

In [ ]:
HTML(JV.animate_walk(best['v'], best['hand']).to_jshtml())

## Step 9. Is it a minimum?  The Hessian
$\mathbf{H} = \partial^2 J / \partial \mathbf{v}^2$ (exact, autograd), scaled so one unit is each number's whole range, with numbers that sit on a bound held out:
- gradient ≈ 0 and all eigenvalues > 0: a **strict local minimum** (a bowl);
- some eigenvalues ≈ 0: a **flat valley**, a family of equally good legs;
- a negative eigenvalue: a **saddle**. The pipeline then restarts along that eigenvector.

The contour map is Step 9 of C13 at the minimum: $J$ on the plane of the two softest eigenvectors, with the LM path projected onto it.

**What the Hessian cannot say:** it certifies a *local* minimum only. The evidence for *global* is the basin table: the best basin reached again from independent LHS/DE runs, and nothing lower found. If $J^* \approx 0$, it is global because $J \geq 0$.

In [ ]:
JO.set_hand(best['hand'])
rep = best.get('rep') or JO.hessian_report(best['v'])
JV.plot_hessian_slice(best['v'], best['hand'], rep, JO.score, best['trace_V']); plt.show()
JO.global_evidence(basins, runs)

## Step 10. One run in all 27 numbers, watched step by step
The best basin's start, replayed through batched LM (every accepted step). The last frame is the `least_squares` finish.

In [ ]:
HTML(JV.animate_lm(best['trace_V'], best['trace_J'], best['hand']).to_jshtml())

## Try it
1. Set `JO.MU_REQ = 0` and rerun from Step 2. Does the optimizer move back toward Jansen's proportions?
2. In Step 9, which numbers sit on a bound (`rep['at_bound']`)? Widen `JO.LO` / `JO.HI` there and rerun Step 6. Did $J^*$ drop?
3. Raise `seeds` in Step 6. Is the best basin reached from more independent runs, or does a new, lower one appear?
4. On the RTX 3090 workstation, after installing a CUDA build of torch: `python jansen_opt_torch.py --bench`. Then use `--screen-device cuda --screen-dtype float32` if it wins.